In [ ]:
# =============================
# IMPORTS
# =============================
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from datetime import datetime, timedelta
import logging
import time

logging.getLogger("yfinance").setLevel(logging.CRITICAL)

# =============================
# SETTINGS
# =============================
LOOKBACK_DAYS = 7

MIN_PRICE = 25
MAX_PRICE = 500

MAX_VOLATILITY = 0.07
MIN_TREND = 0.0

# Filters
USE_REACTION_FILTER = True
USE_SPIKE_FILTER = True

MAX_EARNINGS_MOVE = 0.15  # 15%

# =============================
# DATE SETUP (UPDATED)
# =============================
today = datetime.now().date()

# Adjust for weekends
if today.weekday() == 5:      # Saturday
    today = today - timedelta(days=1)
elif today.weekday() == 6:    # Sunday
    today = today - timedelta(days=2)

yesterday = today - timedelta(days=1)

# Lookback from yesterday (7 days prior)
start_date = yesterday - timedelta(days=LOOKBACK_DAYS - 1)

print(f"\n📅 Earnings window: {start_date} → {yesterday} (excluding today)")
print(f"Today ({today}): Stocks reporting earnings today are EXCLUDED\n")

# =============================
# NASDAQ EARNINGS FETCH
# =============================
def get_earnings_for_day(date):
    url = f"https://api.nasdaq.com/api/calendar/earnings?date={date}"
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        r = requests.get(url, headers=headers)
        data = r.json()
        return [row["symbol"] for row in data["data"]["rows"]]
    except:
        return []

earnings_tickers = []
current = start_date

print("🔎 Fetching earnings from last 7 days (excluding today)...")

while current <= yesterday:
    daily = get_earnings_for_day(current.strftime("%Y-%m-%d"))
    print(f"{current}: {len(daily)} stocks")
    earnings_tickers.extend(daily)
    current += timedelta(days=1)
    time.sleep(0.3)

earnings_tickers = list(set(earnings_tickers))

print(f"\n✅ Total earnings stocks (last 7 days): {len(earnings_tickers)}")

# =============================
# GET TODAY'S EARNINGS & REMOVE THEM
# =============================
todays_earnings = get_earnings_for_day(today.strftime("%Y-%m-%d"))
print(f"\n🚫 Today's earnings ({today}): {len(todays_earnings)} stocks - excluding them")

# Remove any stocks reporting earnings today
earnings_tickers = [s for s in earnings_tickers if s not in todays_earnings]

print(f"✅ Final earnings stocks after removing today's: {len(earnings_tickers)}\n")

# =============================
# FILTERING
# =============================
results = []

counts = {
    "start": len(earnings_tickers),
    "price": 0,
    "trend": 0,
    "volatility": 0,
    "final": 0
}

print("⚙️ Running filters...\n")

for symbol in earnings_tickers:
    try:
        stock = yf.Ticker(symbol)
        hist = stock.history(period="3mo")

        if len(hist) < 50:
            continue

        price = hist["Close"].iloc[-1]

        # PRICE FILTER
        if price < MIN_PRICE or price > MAX_PRICE:
            continue
        counts["price"] += 1

        # EARNINGS REACTION FILTERS
        if USE_REACTION_FILTER or USE_SPIKE_FILTER:
            recent = hist.tail(10)
            returns = recent["Close"].pct_change()
            idx = returns.abs().idxmax()

            try:
                pre_price = hist.loc[:idx].iloc[-2]["Close"]
                post_price = hist.loc[idx]["Close"]
            except:
                continue

            move_pct = (post_price - pre_price) / pre_price

            if USE_SPIKE_FILTER and abs(move_pct) > MAX_EARNINGS_MOVE:
                continue

            if USE_REACTION_FILTER and post_price < pre_price:
                continue

        # TREND FILTER
        hist["SMA20"] = hist["Close"].rolling(20).mean()
        hist["SMA50"] = hist["Close"].rolling(50).mean()

        if pd.isna(hist["SMA50"].iloc[-1]):
            continue

        trend = (hist["SMA20"].iloc[-1] - hist["SMA50"].iloc[-1]) / hist["SMA50"].iloc[-1]

        if trend < MIN_TREND:
            continue
        counts["trend"] += 1

        # VOLATILITY FILTER
        vol = hist["Close"].pct_change().std()

        if vol > MAX_VOLATILITY:
            continue
        counts["volatility"] += 1

        # =============================
        # SPREAD SELECTOR
        # =============================
        spread = 2.5  # default

        strong_trend = trend > 0.03
        clean_reaction = True if not USE_REACTION_FILTER else (post_price >= pre_price)

        if strong_trend and clean_reaction and vol > 0.02:
            spread = 5

        # OUTPUT
        info = stock.info
        sector = info.get("sector", "Unknown")

        results.append({
            "Ticker": symbol,
            "Sector": sector,
            "Price": round(price, 2),
            "Trend %": round(trend * 100, 2),
            "Volatility": round(vol, 4),
            "Suggested Spread": f"${spread}"
        })

        counts["final"] += 1

    except:
        continue

# =============================
# DEBUG & RESULTS
# =============================
print("\n📊 FILTER PROGRESSION:\n")
for k, v in counts.items():
    print(f"{k.capitalize():<12}: {v}")

df = pd.DataFrame(results)

if df.empty:
    print("\n❌ No valid setups this cycle")
else:
    df_sorted = df.sort_values(by="Trend %", ascending=False)

    print("\n✅ TOP CANDIDATES:\n")
    print(df_sorted.head(15).to_string(index=False))

    print("\n📋 COPY LIST:\n")
    print(", ".join(df_sorted["Ticker"].head(15).tolist()))

# Strategy guide remains the same...